# Smart Vest ML Model Training

This notebook generates a physiologically realistic synthetic dataset (1800 samples) and pre-trains the GradientBoostingClassifier on it.

Feature order MUST be alphabetical to match BaselineCalibrator.normalize():
    0  combined_motion
    1  gsr_peak_count
    2  gsr_slope
    3  gsr_std
    4  gyro_variance
    5  hr_mean
    6  hrv_std
    7  imu_peak_count
    8  imu_variance
    9  mean_gsr
    10 mean_hrv
    11 respiration_mean
    12 respiration_variability
    13 resp_zero_crossings

In [ ]:
# Install XGBoost if not already installed
!pip install xgboost joblib numpy

In [ ]:
import numpy as np
import os
import joblib
import xgboost as xgb
from xgboost import XGBClassifier

def _generate_sample(stress_level: float, rng: np.random.Generator) -> list:
    """Synthesise a realistic 14-feature vector for the given stress_level (0–1)."""
    sl = stress_level

    # ── GSR ─────────────────────────────────────────────────────────────────
    mean_gsr       = rng.normal(2.0 + sl * 4.5,          0.3  + sl * 0.4)
    gsr_std        = rng.normal(0.05 + sl * 0.45,         0.02 + sl * 0.05)
    gsr_slope      = rng.normal(-0.001 + sl * 0.016,      0.002)
    gsr_peak_count = max(0.0, rng.normal(sl * 6.5,        0.5  + sl * 1.5))

    # ── HR / HRV ────────────────────────────────────────────────────────────
    hr_mean  = rng.normal(70.0 + sl * 30.0,  2.0 + sl * 3.0)
    mean_hrv = max(5.0, rng.normal(50.0 - sl * 35.0, 2.0 + sl * 2.0))
    hrv_std  = rng.normal(2.0  + sl * 6.5,   0.5)

    # ── IMU ─────────────────────────────────────────────────────────────────
    imu_variance    = max(0.0, rng.normal(0.01 + sl * 0.38,  0.005 + sl * 0.08))
    gyro_variance   = max(0.0, rng.normal(0.003 + sl * 1.3,  0.002 + sl * 0.3))
    combined_motion = rng.normal(1.0 + sl * 2.6,             0.05  + sl * 0.5)
    imu_peak_count  = max(0.0, rng.normal(sl * 4.5,          0.3   + sl * 1.0))

    # ── Respiration ──────────────────────────────────────────────────────────
    respiration_mean        = rng.normal(-0.02 + sl * 0.05, 0.05)
    respiration_variability = rng.normal(0.68 - sl * 0.22,  0.03)
    resp_zero_crossings     = max(0.0, rng.normal(14.0 + sl * 12.0, 1.0 + sl * 2.0))

    # Return in alphabetical order
    return [
        combined_motion,
        gsr_peak_count,
        gsr_slope,
        gsr_std,
        gyro_variance,
        hr_mean,
        hrv_std,
        imu_peak_count,
        imu_variance,
        mean_gsr,
        mean_hrv,
        respiration_mean,
        respiration_variability,
        resp_zero_crossings,
    ]

In [ ]:
rng   = np.random.default_rng(42)
X, y = [], []

# Calm — stress_level 0.00–0.15  → label 0
for _ in range(50000):
    X.append(_generate_sample(rng.uniform(0.00, 0.15), rng))
    y.append(0)

# Low stress — stress_level 0.15–0.40  → label 0
for _ in range(25000):
    X.append(_generate_sample(rng.uniform(0.15, 0.40), rng))
    y.append(0)

# Transitional low — stress_level 0.40–0.58  → label 0
for _ in range(15000):
    X.append(_generate_sample(rng.uniform(0.40, 0.58), rng))
    y.append(0)

# Transitional high — stress_level 0.58–0.72  → label 1
for _ in range(15000):
    X.append(_generate_sample(rng.uniform(0.58, 0.72), rng))
    y.append(1)

# Moderate stress — stress_level 0.72–0.85  → label 1
for _ in range(25000):
    X.append(_generate_sample(rng.uniform(0.72, 0.85), rng))
    y.append(1)

# Full stress — stress_level 0.85–1.00  → label 1
for _ in range(50000):
    X.append(_generate_sample(rng.uniform(0.85, 1.00), rng))
    y.append(1)

X = np.array(X)
y = np.array(y)

# Shuffle before training
idx = rng.permutation(len(y))
X = X[idx]
y = y[idx]

print(f"Generated {len(y)} synthetic samples with 14 features each.")

In [ ]:
model = XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.80,
    tree_method='hist',
    device='cuda', # Use GPU
    random_state=42,
)

print("Training XGBoost Classifier on GPU...")
model.fit(X, y)
print("Training complete.")


In [ ]:
# Save the model to ml_pipeline directory
model_path = "../ml_pipeline/model.pkl"
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")